# Stage 6 — Build the graph and run the core analysis

The analytical heart. Build the directed predicate graph (edge points predicate → citing device),
confirm it is acyclic, then compute the **persistent-recalled-predicate** analysis and all
sensitivity subsets.

Because this corpus (154 codes) differs from the original 9-code study, the headline numbers are
**computed here, not assumed**. This notebook writes `data/expected_values.json` — the analyst's
authoritative results, which Stage 8 (verify) reproduces independently.

Network: none. Fully deterministic — re-run anytime, it always reproduces.

**2026-09 re-run:** the primary edge set is now Rule 2 (`data/predicate_edges.csv`, restricted rule,
DEVIATIONS.md D4). The registered Rule 1 graph (`data/predicate_edges_rule1.csv`) is reported as the
`sens_maximal_rule` sensitivity arm. Tier names are `NEAR_CUE` / `DISTANT_CUE`. Two further arms are
reported: recalled devices whose severity class is `n/a` (`sens_class_na`) and, if `pilot_codes.txt`
exists, the prereg §9 independence panel on the non-pilot product codes.

In [1]:
import os
import json
import pandas as pd
import networkx as nx

corpus = pd.read_csv("data/corpus.csv", dtype=str)
corpus["decision_year"] = pd.to_numeric(corpus["decision_year"], errors="coerce")
edges = pd.read_csv("data/predicate_edges.csv", dtype=str)            # Rule 2 (primary)
edges_rule1 = pd.read_csv("data/predicate_edges_rule1.csv", dtype=str)  # Rule 1 (registered; sensitivity)
recalled = pd.read_csv("data/recalled_nodes.csv", dtype=str)

year = dict(zip(corpus["k_number"], corpus["decision_year"]))
recall_date = dict(zip(recalled["k_number"], pd.to_datetime(recalled["event_date_initiated"], errors="coerce")))
recall_class = dict(zip(recalled["k_number"], recalled["worst_class"]))
recalled_set = set(recalled["k_number"]) & set(corpus["k_number"])
CODES_ALL = json.load(open("snapshot/SNAPSHOT.json"))["product_codes"]

### Build the DiGraph (all corpus devices are nodes)

In [2]:
G = nx.DiGraph()
G.add_nodes_from(corpus["k_number"])
for _, r in edges.iterrows():
    if r["predicate_knumber"] in G and r["device_knumber"] in G:
        G.add_edge(r["predicate_knumber"], r["device_knumber"], confidence=r["confidence"])

is_dag = nx.is_directed_acyclic_graph(G)
connected = [n for n in G if G.degree(n) > 0]
depth = nx.dag_longest_path_length(nx.condensation(G))  # Option A: longest predicate chain via condensation; handles the lone mutual-citation 2-cycle without altering the edge set
largest_wcc = max((len(c) for c in nx.weakly_connected_components(G)), default=0)
print(f"CHECKPOINT  nodes {G.number_of_nodes()} | edges {G.number_of_edges()} | DAG {is_dag}")
print(f"CHECKPOINT  connected {len(connected)} | largest component {largest_wcc} | max depth {depth}")

CHECKPOINT  nodes 15581 | edges 29234 | DAG False
CHECKPOINT  connected 11960 | largest component 11638 | max depth 30


### Recall prevalence

In [3]:
recalled_nodes = recalled_set & set(G.nodes())
pct = round(100 * len(recalled_nodes) / G.number_of_nodes(), 1)
print(f"CHECKPOINT  recalled nodes {len(recalled_nodes)} = {pct}% of all nodes")

CHECKPOINT  recalled nodes 1691 = 10.9% of all nodes


### Persistent recalled predicates

A recalled device A is a *persistent recalled predicate* if a later device B was cleared **after**
A's recall date and B is itself **not** recalled.

In [4]:
def analysis(edge_df, recalled_ok):
    """edge_df: edges to use; recalled_ok: set of recalled predicates to consider."""
    H = nx.DiGraph()
    H.add_nodes_from(corpus["k_number"])
    for _, r in edge_df.iterrows():
        if r["predicate_knumber"] in H and r["device_knumber"] in H:
            H.add_edge(r["predicate_knumber"], r["device_knumber"])
    preds, cits, downstream = set(), 0, set()
    for A in recalled_ok:
        if A not in H:
            continue
        rd = recall_date.get(A)
        if pd.isna(rd):
            continue
        for B in H.successors(A):
            if B in recalled_set:
                continue
            by = year.get(B)
            if pd.notna(by) and by > rd.year:
                preds.add(A); cits += 1; downstream.add(B)
    return len(preds), cits, len(downstream)

base = analysis(edges, recalled_nodes)
print(f"CHECKPOINT  persistent predicates {base[0]} | citations {base[1]} | downstream {base[2]}")

# first cited only AFTER own recall
first_after = 0
for A in recalled_nodes:
    rd = recall_date.get(A)
    if A not in G or pd.isna(rd):
        continue
    citer_years = [year.get(B) for B in G.successors(A) if pd.notna(year.get(B))]
    if citer_years and min(citer_years) > rd.year:
        first_after += 1
print(f"CHECKPOINT  first cited AFTER own recall {first_after}")

# max recall-to-latest-citation gap
max_gap = 0.0
for A in recalled_nodes:
    rd = recall_date.get(A)
    if A not in G or pd.isna(rd):
        continue
    cyrs = [year.get(B) for B in G.successors(A) if pd.notna(year.get(B))]
    if cyrs:
        max_gap = max(max_gap, max(cyrs) - rd.year)
print(f"CHECKPOINT  max recall-to-latest gap {max_gap:.1f} yr")

CHECKPOINT  persistent predicates 921 | citations 2712 | downstream 1798
CHECKPOINT  first cited AFTER own recall 382
CHECKPOINT  max recall-to-latest gap 21.0 yr


### Sensitivity subsets

In [5]:
# Class I/II only
cii = {A for A in recalled_nodes if recall_class.get(A) in ("Class I", "Class II")}
s_cii = analysis(edges, cii)
# high-confidence edges only
hc_edges = edges[edges["confidence"] == "NEAR_CUE"]   # was SECTION_HEADED
s_hc = analysis(hc_edges, recalled_nodes)
# combined
s_comb = analysis(hc_edges, cii)
print(f"CHECKPOINT  class_I_II {s_cii} | high_conf {s_hc} | combined {s_comb}")

# --- post hoc arms (DEVIATIONS.md D6) ---
# registered maximal rule (Rule 1) — what the graph looked like before the restriction
s_max = analysis(edges_rule1, recalled_nodes)
# recalled devices with NO severity class (excluded from sens_class_I_II)
cna = {A for A in recalled_nodes if recall_class.get(A) not in ("Class I", "Class II", "Class III")}
s_cna = analysis(edges, cna)
n_class = {c: sum(1 for A in recalled_nodes if recall_class.get(A) == c) for c in ("Class I", "Class II", "Class III")}
n_class["n/a"] = len(cna)
print(f"CHECKPOINT  maximal_rule {s_max} | class_na {s_cna} | recalled by class {n_class}")

# --- prereg §9 independence panel: non-pilot product codes only (optional) ---
panel = None
if os.path.exists("pilot_codes.txt"):
    PILOT = {c.strip().upper() for c in open("pilot_codes.txt") if c.strip() and not c.startswith("#")}
    is_pilot = corpus["product_codes"].fillna("").apply(lambda s: bool(set(s.split("|")) & PILOT))
    nonpilot = set(corpus.loc[~is_pilot, "k_number"])
    np_edges = edges[edges["predicate_knumber"].isin(nonpilot) & edges["device_knumber"].isin(nonpilot)]
    np_rec = recalled_nodes & nonpilot
    P = nx.DiGraph(); P.add_nodes_from(nonpilot)
    for _, r in np_edges.iterrows():
        P.add_edge(r["predicate_knumber"], r["device_knumber"])
    fa, mg = 0, 0.0
    for A in np_rec:
        rd = recall_date.get(A)
        if A not in P or pd.isna(rd): continue
        cy = [year.get(B) for B in P.successors(A) if pd.notna(year.get(B))]
        if cy:
            if min(cy) > rd.year: fa += 1
            mg = max(mg, max(cy) - rd.year)
    panel = {"n_pilot_codes": len(PILOT), "n_nonpilot_codes": len(set(CODES_ALL) - PILOT),
             "devices": len(nonpilot), "edges": int(len(np_edges)), "recalled_nodes": len(np_rec),
             "persistent_predicates": analysis(np_edges, np_rec)[0],
             "post_recall_citations": analysis(np_edges, np_rec)[1],
             "downstream_devices": analysis(np_edges, np_rec)[2],
             "first_after_recall": fa, "max_gap_years": round(mg, 1)}
    print(f"CHECKPOINT  §9 non-pilot panel {panel}")
else:
    print("NOTE  pilot_codes.txt not found — §9 non-pilot panel skipped (add the 9 pilot product codes, one per line, to run it)")

CHECKPOINT  class_I_II (703, 1997, 1361) | high_conf (816, 2087, 1496) | combined (634, 1554, 1149)
CHECKPOINT  maximal_rule (938, 2877, 1873) | class_na (210, 704, 583) | recalled by class {'Class I': 11, 'Class II': 1306, 'Class III': 18, 'n/a': 356}
NOTE  pilot_codes.txt not found — §9 non-pilot panel skipped (add the 9 pilot product codes, one per line, to run it)


### Write the authoritative expected values (for Stage 8)

In [6]:
cov = pd.read_csv("data/coverage_diagnostic.csv")
covc = cov["coverage"].value_counts(); ncov = len(cov)
EXPECTED = {
    "snapshot_date": json.load(open("snapshot/SNAPSHOT.json"))["snapshot_date"],
    "corpus_devices": int(len(corpus)),
    "edges": int(G.number_of_edges()),
    "is_dag": bool(is_dag),
    "connected_nodes": int(len(connected)),
    "max_chain_depth": int(depth),
    "largest_component": int(largest_wcc),
    "recalled_nodes": int(len(recalled_nodes)),
    "pct_recalled": pct,
    "persistent_predicates": base[0], "post_recall_citations": base[1], "downstream_devices": base[2],
    "first_after_recall": first_after,
    "max_gap_years": round(max_gap, 1),
    "sens_class_I_II": list(s_cii),
    "sens_high_conf": list(s_hc),
    "sens_combined": list(s_comb),
    # --- added 2026-09 (D4/D6) ---
    "edge_rule": "rule2_restricted",
    "edges_rule1": int(len(edges_rule1.drop_duplicates(subset=["predicate_knumber","device_knumber"]))),
    "edges_excluded_by_rule2": int(len(pd.read_csv("data/excluded_edges.csv", dtype=str))),
    "sens_maximal_rule": list(s_max),
    "sens_class_na": list(s_cna),
    "recalled_by_class": n_class,
    "panel_nonpilot": panel,
    "cov_excluded_only_pct": round(100*covc.get("excluded_only",0)/ncov, 1),
    "text_corpus_sha256": json.load(open("TEXT_SNAPSHOT.json"))["corpus_sha256"],
    "cov_resolvable_pct": round(100*covc.get("edge_resolvable",0)/ncov, 1),
    "cov_name_only_pct": round(100*covc.get("name_only",0)/ncov, 1),
    "cov_out_of_scope_pct": round(100*covc.get("out_of_scope",0)/ncov, 1),
}
json.dump(EXPECTED, open("data/expected_values.json", "w"), indent=2)
nx.write_gml(G, "data/predicate_graph.gml")
print("CHECKPOINT  wrote data/expected_values.json and data/predicate_graph.gml")
print(json.dumps(EXPECTED, indent=2))

CHECKPOINT  wrote data/expected_values.json and data/predicate_graph.gml
{
  "snapshot_date": "2026-08-08",
  "corpus_devices": 15581,
  "edges": 29234,
  "is_dag": false,
  "connected_nodes": 11960,
  "max_chain_depth": 30,
  "largest_component": 11638,
  "recalled_nodes": 1691,
  "pct_recalled": 10.9,
  "persistent_predicates": 921,
  "post_recall_citations": 2712,
  "downstream_devices": 1798,
  "first_after_recall": 382,
  "max_gap_years": 21,
  "sens_class_I_II": [
    703,
    1997,
    1361
  ],
  "sens_high_conf": [
    816,
    2087,
    1496
  ],
  "sens_combined": [
    634,
    1554,
    1149
  ],
  "edge_rule": "rule2_restricted",
  "edges_rule1": 30476,
  "edges_excluded_by_rule2": 1242,
  "sens_maximal_rule": [
    938,
    2877,
    1873
  ],
  "sens_class_na": [
    210,
    704,
    583
  ],
  "recalled_by_class": {
    "Class I": 11,
    "Class II": 1306,
    "Class III": 18,
    "n/a": 356
  },
  "panel_nonpilot": null,
  "cov_excluded_only_pct": 0.3,
  "text_corpus